# 09 · Text-to-Video

> **Source notes:** `TextToVideo.md`

Video = images + time. Making those frames consistent is the hard part.

This notebook (CPU-friendly, no downloads required):
- Represents a **video tensor** `(T,C,H,W)` and measures frame-to-frame consistency
- Builds a **temporal self-attention** module from scratch and shows how it ties frames together
- Compares coherence of independent-frame generation vs. temporally-aware generation on synthetic sequences
- Displays an AnimateDiff model architecture summary from `diffusers` metadata (inspection only)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Plot results -- call `run()`
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "matplotlib", "numpy", "-q"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
print("Ready.")

## 1 · Video Tensor Basics

In [ ]:
def frame_consistency(video):
    """
    TODO #2: Implement `frame_consistency()`.

    Steps:
    1. Process data
    2. Compute `independent_video` using `manual_seed()`
    3. Compute `key1` using `rand()`
    4. Define helper function `frame_consistency()`
    5. Compute `diffs_indep`
    6. Plot results -- call `plot()`
    7. Plot results -- call `permute()`
    8. Plot results -- call `suptitle()`
    9. Call `mean()` to produce the result

    Hint:
    independent_video = torch.rand(???)
    key1 = torch.rand(???)
    key2 = torch.rand(???)
    lambdas = torch.linspace(???)

    Returns: diffs
    """
    raise NotImplementedError("TODO: implement frame_consistency()")

## 2 · Temporal Self-Attention From Scratch

At each spatial position, attend across the T frames. This is the core of AnimateDiff's temporal attention modules.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define helper function
# 2. Call `qkv()` to produce the result
# 3. Call `transpose()` to produce the result
# 4. Call `reshape()` to produce the result
# 5. Compute `temp_attn` using `TemporalAttention()`
#
# Hint:
#    temp_attn = TemporalAttention(channels=???)
#    qkv = nn.Linear(???)
#    proj = nn.Linear(???)
#    pos_embed = nn.Parameter(???)

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
class TemporalAttention(nn.Module):
    """
    Self-attention across the temporal dimension.
    Input:  (B, T, C, H, W)
    Output: (B, T, C, H, W)  — each frame position informed by all other frames
    """
    def __init__(self, channels, n_heads=4):
        super().__init__()
        assert channels % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = channels // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv  = nn.Linear(channels, channels * 3)
        self.proj = nn.Linear(channels, channels)
        # Positional embedding for T frames
        self.pos_embed = nn.Parameter(torch.zeros(1, 16, channels))  # max 16 frames
        nn.init.normal_(self.pos_embed, std=0.02)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        # Reshape: (B*H*W, T, C) — attend across T at each spatial position
        x_flat = x.permute(0, 3, 4, 1, 2).reshape(B*H*W, T, C)
        x_flat = x_flat + self.pos_embed[:, :T, :]  # add temporal position encoding
        
        qkv = self.qkv(x_flat).reshape(B*H*W, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B*H*W, heads, T, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn  = (q @ k.transpose(-2,-1)) * self.scale
        attn  = attn.softmax(dim=-1)
        out   = (attn @ v).transpose(1, 2).reshape(B*H*W, T, C)
        out   = self.proj(out)
        
        # Reshape back: (B*H*W, T, C) → (B, T, C, H, W)
        out   = out.reshape(B, H, W, T, C).permute(0, 3, 4, 1, 2)
        return out


temp_attn = TemporalAttention(channels=32)
x_in      = torch.randn(2, 8, 32, 8, 8)   # (batch=2, T=8, C=32, H=8, W=8)
x_out     = temp_attn(x_in)
print(f"Input  shape: {x_in.shape}")
print(f"Output shape: {x_out.shape}")
print(f"Temporal block params: {sum(p.numel() for p in temp_attn.parameters()):,}")

## 3 · Visualise Temporal Attention Weights

Show which frames each frame attends to — before and after learning.

In [ ]:
def get_attn_weights(module, x):
    """
    TODO #4: Implement `get_attn_weights()`.

    Steps:
    1. Define helper function `get_attn_weights()`
    2. Compute `T_frames` using `randn()`
    3. Compute `module_random` using `Untrained()`
    4. Compute `module_trained` using `attention()`
    5. Plot results -- call `subplots()`
    6. Plot results -- call `imshow()`
    7. Plot results -- call `suptitle()`

    Hint:
    module_random = TemporalAttention(channels=???)
    module_trained = TemporalAttention(channels=???)
    x_flat = x.permute(???)
    qkv = module.qkv(???)

    Returns: attn[0, 0].detach().numpy()   # first...
    """
    raise NotImplementedError("TODO: implement get_attn_weights()")

## 4 · AnimateDiff Architecture Summary (inspection, no download)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call `Architecture()` to produce the result
# 2. Call `Net()` to produce the result
# 3. Call `CrossAttention()` to produce the result
# 4. Call `3C()` to produce the result
# 5. Call `to()` to produce the result
# 6. Process data
# 7. Call `5()` to produce the result
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
print("""
AnimateDiff Architecture (from the paper / diffusers implementation)
=====================================================================

Base: Stable Diffusion 1.5 U-Net (FROZEN during motion module training)

For each SD U-Net block:
  [ResBlock] → [SpatialSelfAttention] → [TemporalAttention ← TRAINED] → [CrossAttention(text)]

TemporalAttention parameters per block:
  - QKV projection: C × 3C   (e.g., 320 × 960 = 307,200)
  - Output projection: C × C
  - Temporal positional embedding: T × C
  Total motion module: ~350 MB (for C=320 backbone)

Sampling flow:
  1. Random latent noise: (B, T, 4, 64, 64)
  2. DDIM 20 steps in latent space:
     - Spatial layers process each frame: (B*T, 4, 64, 64)
     - Temporal layers bridge frames:     (B, T, 4, 64, 64)
     - Cross-attention with CLIP text: same text for all frames
  3. VAE decode each frame: (B*T, 3, 512, 512)
  4. Reshape to (B, T, 3, 512, 512) → save as GIF / MP4

Why style LoRAs work with AnimateDiff:
  - Spatial SD weights are frozen → swappable like regular SD
  - LoRA patches the Q/K/V projections of spatial attention
  - Temporal attention is unaffected by spatial LoRAs
  → Load 'anime style' LoRA → AnimateDiff generates anime-style video

Memory on a modern GPU:
  SD 1.5 (spatial): ~2 GB fp16
  Motion module:    ~350 MB fp16
  VAE:              ~300 MB fp16
  KV cache (T=16):  ~800 MB fp16
  Total:            ~3.5 GB — fits in 6 GB VRAM
""")

## 5 · Summary

```
Key T2V takeaways:

 Problem: Videos have a time axis. Independent frame generation = flicker.

 Solution: temporal attention: at position (h,w), attend across T frames.

 AnimateDiff: freeze SD spatial layers + train only temporal attention
 on video-text pairs → 350 MB motion module, plug into any SD.

 Sora/CogVideoX: fully 3D architecture from the start
 spacetime patches as tokens → arbitrary resolution & duration.

 Cost: 16 frames at 512×512 ≈ 16× the cost of one SD image.
 DDIM 20 steps × 16 frames = 320 U-Net forward passes.
```

**Next:** [MultimodalLLMs.md](../MultimodalLLMs/MultimodalLLMs.md) — understand images not just generate them.